Ответ: 56573.48 37194.91

# 1. Загрузите данные об описаниях вакансий и соответствующих годовых зарплатах из файла salary-train.csv.

In [14]:
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
import re
from scipy.sparse import hstack

data_train = pd.read_csv("/content/salary-train.csv")
data_test = pd.read_csv("/content/salary-test-mini.csv")

# 2. Проведите предобработку: • Приведите тексты к нижнему регистру. • Замените все, кроме букв и цифр, на пробелы — это облегчит дальнейшее разделение текста на слова. Для такой замены в строке text подходит следующий вызов: r e . sub ( ’ [^ a−zA−Z0−9] ’ , ’ ␣ ’ , t e x t . low e r ( ) ) • Примените TfidfVectorizer для преобразования текстов в векторы признаков. Оставьте только те слова, которые встречаются хотя бы в 5 объектах (параметр min_df у TfidfVectorizer). • Замените пропуски в столбцах LocationNormalized и ContractTime на специальную строку ’nan’. Код для этого был приведен выше. • Примените DictVectorizer для получения one-hot-кодирования признаков LocationNormalized и ContractTime. • Объедините все полученные признаки в одну матрицу "объектыпризнаки". Обратите внимание, что матрицы для текстов и категориальных признаков являются разреженными. Для объединения их столбцов нужно воспользоваться функцией scipy.sparse.hstack.


In [16]:
# Заполнение пропусков
data_train['LocationNormalized'] = data_train['LocationNormalized'].fillna('nan')
data_train['ContractTime'] = data_train['ContractTime'].fillna('nan')
data_test['LocationNormalized'] = data_test['LocationNormalized'].fillna('nan')
data_test['ContractTime'] = data_test['ContractTime'].fillna('nan')

# Очистка текста
data_train['FullDescription'] = data_train['FullDescription'].str.lower().str.replace(r'[^a-zA-Z0-9]', ' ', regex=True)
data_test['FullDescription'] = data_test['FullDescription'].str.lower().str.replace(r'[^a-zA-Z0-9]', ' ', regex=True)

# TF-IDF
vectorizer = TfidfVectorizer(min_df=5)
data_train_tfidf = vectorizer.fit_transform(data_train['FullDescription'])
data_test_tfidf = vectorizer.transform(data_test['FullDescription'])

# DictVectorizer
enc = DictVectorizer()
X_train_categ = enc.fit_transform(data_train[['LocationNormalized', 'ContractTime']].to_dict('records'))
X_test_categ = enc.transform(data_test[['LocationNormalized', 'ContractTime']].to_dict('records'))

# Объединение матриц
X_train = hstack([data_train_tfidf, X_train_categ])
X_test = hstack([data_test_tfidf, X_test_categ])

# 3. Обучите гребневую регрессию с параметром alpha=1. Целевая переменная записана в столбце SalaryNormalized.

In [17]:
y_train = data_train['SalaryNormalized']
model = Ridge(alpha=1, random_state=241)
model.fit(X_train, y_train)

Ridge(alpha=1, random_state=241)

# 4. Постройте прогнозы для двух примеров из файла salary-test-mini.csv. Значения полученных прогнозов являются ответом на задание. Укажите их через пробел.


In [18]:
pred = model.predict(X_test)
print(round(pred[0], 2), round(pred[1], 2))

56573.48 37194.91
